## Prerequisites

Make Redshift Cluster or Redshift Serverless publicly accessible.

Add Elastic IP to Redshift Cluster.

Allow connections to 5439 (redshift) in the associated security group of the cluster

## Setup Redshift Database and User

```sql
CREATE DATABASE retail_db;

CREATE USER retail_user WITH PASSWORD 'Itv3rs1ty';

GRANT ALL ON DATABASE retail_db TO retail_user;

/*
Host: default-workgroup.222634372385.ap-southeast-2.redshift-serverless.amazonaws.com
Port: 5439
Database: retail_db
User: retail_user
Password Itv3rs1ty

*/
```

Use psql to connect to the Redshift cluster
```bash
psql -h default-workgroup.222634372385.ap-southeast-2.redshift-serverless.amazonaws.com \
-p 5439 \
-d retail_db \
-U retail_user \
-W

# get table information
\d

SELECT * FROM orders LIMIT 10;
# might need to change owner of orders table to retail_user

```

Go back to Redshift Query Editor and run:
```sql
ALTER TABLE orders OWNER to retail_db
```

Go back to psql connected to the Redshift cluster, the query should work now:
```sql
SELECT * FROM orders LIMIT 10;
```


## Connecting to Redshift Databases using an IDE

Download the JDBC Driver Jar file from the Redshift Cluster page.


## Connect to Redshift Databases using Python

Python modules:
- pip install psycopg2-binary

In [1]:
!pip install psycopg2-binary

In [2]:
import os
# os.environ.setdefault('http_proxy', 'http://webproxy.au.harveynorman.com:8080')
# os.environ.setdefault('https_proxy', 'http://webproxy.au.harveynorman.com:8080')
os.environ.setdefault('AWS_DEFAULT_REGION', 'ap-southeast-2')

'ap-southeast-2'

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host='default-workgroup.222634372385.ap-southeast-2.redshift-serverless.amazonaws.com',
    port=5439,
    database='retail_db',
    user='retail_user',
    password='Itv3rs1ty'
)

In [ ]:
cursor = conn.cursor()

In [ ]:
query_str = 'SELECT * FROM orders LIMIT 10;'

In [ ]:
cursor.execute(query_str)
for rec in cursor:
    print(rec)

In [ ]:
cursor.close()

In [ ]:
truncate_stmt = 'TRUNCATE TABLE order_items;'

In [ ]:
cursor = conn.cursor()
cursor.execute(truncate_stmt)

## Connecting to Redshift using boto3 and Copying data

In [ ]:
access_key = 'aws user access key'
secret_key = 'aws user secret key'

import os

os.environ.setdefault('AWS_ACCESS_KEY_ID', access_key)
os.environ.setdefault('AWS_SECRET_ACCESS_KEY', secret_key)

In [ ]:
import boto3

s3_client = boto3.client('s3')

In [ ]:
s3_objects = s3_client.list_objects(
    Bucket='itv-retail',
    Prefix='retail_db_json/'
)

In [ ]:
[obj['Key'] for obj in s3_objects['Contents']]

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host='default-workgroup.222634372385.ap-southeast-2.redshift-serverless.amazonaws.com',
    port=5439,
    database='retail_db',
    user='retail_user',
    password='Itv3rs1ty'
)

In [ ]:
table_name = 'order_items'
s3_location = 's3://itv-retail/retail_db_json/order_items/'

In [ ]:
copy_stmt = f"""
COPY {table_name}
FROM '{s3_location}'
CREDENTIALS 'aws_access_key_id={access_key};aws_secret_access_key={secret_key}'
JSON AS 'auto'
"""

In [ ]:
cursor = conn.cursor()

In [ ]:
cursor.execute(copy_stmt)

In [ ]:
query_stmt = 'SELECT COUNT(*) FROM order_items;'
cursor.execute(query_stmt)

In [ ]:
cursor.fetchone()

In [ ]:
query_stmt = 'SELECT * FROM order_items LIMIT 10;'
cursor.execute(query_stmt)

In [ ]:
cursor.fetchall()